<a href="https://colab.research.google.com/github/divay0407/FakeNewsClassifier/blob/main/FakenewsLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

In [ ]:
df=pd.read_csv("/content/mail_data.csv")

In [ ]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.isnull().sum()

,0
Category,0
Message,0


In [ ]:
x=df.drop("Category",axis=1)

In [ ]:
y=df["Category"]

In [ ]:
x.shape
y.shape

(5572,)

In [ ]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout


In [ ]:
voc_size = 5000

In [ ]:
messages=x.copy()

In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

nltk.download("stopwords")

ps = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
corpus = []

# Clean every SMS message
for i in range(len(messages)):

    # Keep only letters
    review = re.sub("[^a-zA-Z]", " ", messages["Message"][i])

    # Convert to lowercase
    review = review.lower()

    # Split sentence into words
    review = review.split()

    # Remove stopwords and apply stemming
    review = [
        ps.stem(word)
        for word in review
        if word not in stopwords.words("english")
    ]

    # Convert words back into sentence
    review = " ".join(review)

    corpus.append(review)

In [ ]:
print(corpus[:5])

['go jurong point crazi avail bugi n great world la e buffet cine got amor wat', 'ok lar joke wif u oni', 'free entri wkli comp win fa cup final tkt st may text fa receiv entri question std txt rate c appli', 'u dun say earli hor u c alreadi say', 'nah think goe usf live around though']


In [ ]:
onehot_repr = [one_hot(words, voc_size) for words in corpus]

print(onehot_repr[:2])


[[585, 1882, 257, 20, 720, 1088, 3231, 807, 835, 2967, 1095, 3733, 164, 4684, 3909, 3032], [891, 4761, 23, 2939, 446, 1645]]


In [ ]:
#padding

sent_length = 20

embedded_docs = pad_sequences(
    onehot_repr,
    padding="post",
    maxlen=sent_length
)

In [ ]:
print(embedded_docs.shape)

(5572, 20)


In [ ]:
y_final = pd.get_dummies(y, drop_first=True)

In [ ]:
x_final = np.array(embedded_docs)

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_final,
    y_final,
    test_size=0.33,
    random_state=42
)


In [ ]:
embedding_vector_features = 40

model = Sequential()

model.add(
    Embedding(
        input_dim=voc_size,
        output_dim=embedding_vector_features,
        input_length=sent_length
    )
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.add(Dropout(0.3))

model.add(LSTM(100))

model.add(Dropout(0.3))

model.add(Dense(1, activation="sigmoid"))


In [ ]:
# Binary classification problem:
# ham vs spam
#
# Therefore:
# loss = binary_crossentropy

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


In [ ]:
history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=10, batch_size=64)


Epoch 1/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.8979 - loss: 0.2804 - val_accuracy: 0.9657 - val_loss: 0.1058
Epoch 2/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9802 - loss: 0.0724 - val_accuracy: 0.9788 - val_loss: 0.0672
Epoch 3/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9912 - loss: 0.0373 - val_accuracy: 0.9793 - val_loss: 0.0703
Epoch 4/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9949 - loss: 0.0241 - val_accuracy: 0.9728 - val_loss: 0.0980
Epoch 5/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9949 - loss: 0.0180 - val_accuracy: 0.9821 - val_loss: 0.0717
Epoch 6/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9971 - loss: 0.0109 - val_accuracy: 0.9793 - val_loss: 0.1262
Epoch 7/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9960 - loss: 0.0156 - val_accuracy: 0.9772 - val_loss: 0.1043
Epoch 8/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9987 - loss: 0.0081 - val_accuracy: 0.9782 - val_l

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 40)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100)            │        56,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 769,505 (2.94 MB)

 Trainable params: 256,501 (1001.96 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 513,004 (1.96 MB)

In [ ]:
y_pred = model.predict(x_test)

# Convert probabilities into classes
#
# > 0.6  -> spam
# <= 0.6 -> ham

y_pred = np.where(y_pred > 0.6, 1, 0)


58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [ ]:
# EVALUATION
# ============================================================

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))



Confusion Matrix
[[1581   12]
 [  30  216]]

Accuracy
0.9771615008156607

Classification Report
              precision    recall  f1-score   support

       False       0.98      0.99      0.99      1593
        True       0.95      0.88      0.91       246

    accuracy                           0.98      1839
   macro avg       0.96      0.94      0.95      1839
weighted avg       0.98      0.98      0.98      1839



In [ ]:
def predict_message(message, model):
    review = re.sub("[^a-zA-Z]", " ", message)
    review = review.lower()
    review = review.split()
    review = [
        ps.stem(word)
        for word in review
        if word not in stopwords.words("english")
    ]
    review = " ".join(review)

    onehot_encoded = one_hot(review, voc_size)
    padded_sequence = pad_sequences(
        [onehot_encoded],
        padding="post",
        maxlen=sent_length
    )

    prediction = model.predict(padded_sequence)

    if prediction[0][0] > 0.6:
        return "Spam"
    else:
        return "Ham"

predict_message("Hey divay ,can you please reply to my message", model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


'Ham'